<a href="https://colab.research.google.com/github/getcommunityone/open-navigator/blob/main/scripts/datasources/wikidata/load_jurisdictions_wikidata_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wikidata jurisdictions → PostgreSQL

Runs **`load_jurisdictions_wikidata.py`** (WDQS → bronze `*_wikidata`).

## Run and forget (recommended)

From repo root on a laptop or VPS (six **priority states**: AL, GA, IN, MA, WA, WI — every jurisdiction type):

```bash
./scripts/datasources/wikidata/run_wikidata_priority_states_background.sh
tail -f data/logs/wikidata_priority_*.log
```

See **`scripts/datasources/wikidata/README.md`** for env tuning. Use `RUN_FOREGROUND=1 …` if you want the process attached to your terminal.

## Notebook path (interactive)

Run order:

1–2 · Repo root → **3 · Configure** → **4 · DB / env** → **5 · pip** → **6 · Run** (subprocess; logs appear below **`Command:`**).

Long runs hit WDQS throttles and backoff; checkpoints resume under **`WIKIDATA_LOAD_CHECKPOINT_FILE`** (`data/cache/wikidata/` by default). Step **3** defaults mirror the shell runner (**priority** USPS + all types).

## 1 · Repo root (local) or Drive (Colab)

- **Local:** leave `PROJECT_ROOT_MANUAL` empty; the next cell walks up `Path.cwd()` to find `scripts/datasources/wikidata/load_jurisdictions_wikidata.py`.
- **Colab + Drive:** set `PROJECT_ROOT_MANUAL`, or clone in step 2.
- **Colab only:** step 2 clones to `/content/open-navigator`.

In [1]:
from pathlib import Path

PROJECT_ROOT_MANUAL = ""


def _detect_open_navigator_root():
    start = Path.cwd().resolve()
    marker = Path("scripts/datasources/wikidata/load_jurisdictions_wikidata.py")
    for p in [start, *start.parents]:
        if (p / marker).is_file():
            return p
    return None


if PROJECT_ROOT_MANUAL.strip():
    PROJECT_ROOT = PROJECT_ROOT_MANUAL.strip().rstrip("/")
else:
    found = _detect_open_navigator_root()
    PROJECT_ROOT = str(found) if found else ""

## 2 · Get the codebase (Colab if step 1 found no repo)

In [2]:
import os
import subprocess

REPO_URL = "https://github.com/getcommunityone/open-navigator.git"
CLONE_DEST = "/content/open-navigator"


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _wikidata_pkg(p: str) -> str:
    return os.path.join(p, "scripts", "datasources", "wikidata")


if PROJECT_ROOT:
    ROOT = PROJECT_ROOT
    if not os.path.isdir(_wikidata_pkg(ROOT)):
        raise FileNotFoundError("Not an open-navigator root: " + ROOT)
elif _in_colab():
    subprocess.run(["rm", "-rf", CLONE_DEST], check=False)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, CLONE_DEST],
        check=True,
    )
    ROOT = CLONE_DEST
else:
    raise RuntimeError(
        "Repo not found from cwd (no scripts/datasources/wikidata). "
        "Use repo root cwd, set PROJECT_ROOT_MANUAL, or Colab."
    )

os.chdir(ROOT)

## 3 · Configure what to load (**edit here first**)

Values here drive **step 6**. Edit and run this cell after the repo is on disk (no extra packages).

In [ ]:
# ═══ Wikidata loader choices (used in step 6; matches run_wikidata_priority_states_background.sh) ═══
STATES_MODE = "priority"  # explicit | priority | all_us — priority = AL,GA,IN,MA,MT,WA,WI
STATES_CSV = "AL,GA,IN,MA,MT,WA,WI"  # when STATES_MODE == "explicit"
LOAD_TYPES = ["state", "city", "county", "school_district"]

COUNTY_GAP_DISCOVERY = False   # True only with "county" in LOAD_TYPES (Postgres gap list)
IGNORE_CHECKPOINT = False      # True → --force

## 4 · Database URL + env (checkpoint)

The loader resolves Postgres as **`NEON_DATABASE_URL_DEV` → `NEON_DATABASE_URL` → localhost default** (see `load_jurisdictions_wikidata.py` after `load_dotenv()`).

**Local:** Put your URL in **`.env`** and leave **`NEON_URL`** empty below (the subprocess inherits the repo cwd and loads `.env`). Optionally uncomment **`NEON_URL = os.environ.get(...)`** to reuse a shell-exported URL.

If you ever use **Colab**, you may keep the optional **`userdata.get("NEON_DATABASE_URL")`** block.

Next cell applies **run-and-forget-style** WDQS defaults (throttle ~10 s, Retry-After cap 120 s, **incremental merge** on); step **6** still passes **`--priority-states`** / types via CLI.

In [4]:
import os
from pathlib import Path

NEON_URL = ""
try:
    from google.colab import userdata

    NEON_URL = (userdata.get("NEON_DATABASE_URL") or userdata.get("NEON_DATABASE_URL_DEV") or "").strip()
except Exception:
    pass

# Local: pull from the notebook kernel env (same as your shell inherited by Jupyter).
# Uncomment if you export NEON_DATABASE_URL before starting Jupyter instead of using .env.
# NEON_URL = os.environ.get("NEON_DATABASE_URL_DEV", "") or os.environ.get("NEON_DATABASE_URL", "")

# Rare: paste a URL string here if you refuse Colab Secrets and .env.
# NEON_URL = "postgresql://..."

if NEON_URL.strip():
    os.environ["NEON_DATABASE_URL"] = NEON_URL.strip()
    os.environ["NEON_DATABASE_URL_DEV"] = NEON_URL.strip()

# Else: subprocess still runs load_dotenv(); repo .env fills NEON_* if unset in the loader process.

os.environ.setdefault("WIKIDATA_THROTTLE_SECONDS", "10")
os.environ.setdefault("WIKIDATA_TASK_SLEEP_SECONDS", "4")
os.environ.setdefault("WIKIDATA_RETRY_AFTER_MAX_SECONDS", "120")
os.environ.setdefault("WIKIDATA_INCREMENTAL_MERGE", "1")
os.environ.setdefault("WIKIDATA_CACHE_TTL_SECONDS", "604800")

os.environ.setdefault("WIKIDATA_LOAD_ALL_US_STATES", "0")
os.environ.setdefault("WIKIDATA_LOAD_RETRY_COUNTY_GAPS", "0")
os.environ.setdefault("WIKIDATA_LOAD_FORCE", "0")

_root = Path.cwd().resolve()
_cache_rel = (os.environ.get("WIKIDATA_CACHE_DIR") or "data/cache/wikidata").strip()
_ck_dir = _root / _cache_rel
_ck_dir.mkdir(parents=True, exist_ok=True)
os.environ["WIKIDATA_LOAD_CHECKPOINT_FILE"] = str(_ck_dir / "wikidata_jurisdictions_checkpoint.json")

## 5 · Dependencies (`pip`)

In [5]:
import subprocess
import sys

pkgs = ["psycopg2-binary", "httpx", "loguru", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

0

## 6 · Run loader (foreground in this notebook)

Runs **step 3** CLI flags + env from step **4**.

For unattended runs, prefer **`run_wikidata_priority_states_background.sh`** (writes **`data/logs/wikidata_priority_*.log`**).

Interactive: watch **`Command:`** and **`loguru`** lines below — long WDQS-bound jobs are normal. **County gap** needs **`county`** in types. **IGNORE_CHECKPOINT** = `--force`.

In [ ]:
import os
import shutil
import subprocess
import sys

# Requires step **3 · Configure** (`STATES_MODE`, `STATES_CSV`, `LOAD_TYPES`, …).

def build_wikidata_extra_args(states_mode, states_csv, load_types_sel, county_gap_discovery, ignore_checkpoint):
    allowed = frozenset({"state", "city", "town", "county", "school_district"})
    sel = [str(x).strip().lower() for x in tuple(load_types_sel) if str(x).strip()]
    unknown = sorted(set(sel) - allowed)
    if unknown:
        raise ValueError(f"Unknown types {unknown}; allowed {sorted(allowed)}")
    if not sel:
        raise ValueError("Select at least one jurisdiction type.")

    extra: list[str] = ["--types", ",".join(sel)]

    if county_gap_discovery:
        if "county" not in sel:
            raise ValueError('County-gap mode requires "county" in types.')
        extra += [
            "--retry-county-gap-states",
            "--no-priority-states",
            "--no-all-us-states",
        ]
    elif states_mode == "explicit":
        sc = str(states_csv).replace(" ", "").strip().upper()
        if not sc:
            raise ValueError("STATES CSV empty (need e.g. AL or AL,TX)")
        extra += [
            "--no-retry-county-gap-states",
            "--no-priority-states",
            "--no-all-us-states",
            "--states",
            sc,
        ]
    elif states_mode == "priority":
        extra += [
            "--no-retry-county-gap-states",
            "--no-all-us-states",
            "--priority-states",
        ]
    elif states_mode == "all_us":
        extra += [
            "--no-retry-county-gap-states",
            "--no-priority-states",
            "--all-us-states",
        ]
    else:
        raise ValueError("states_mode must be explicit | priority | all_us")

    extra.append("--force" if ignore_checkpoint else "--no-force")
    return extra


def launch_loader(extra):
    """Print full command and invoke loader (inherits os.environ including step 4)."""
    root = os.getcwd()
    script = os.path.join(root, "scripts", "datasources", "wikidata", "load_jurisdictions_wikidata.py")
    if not os.path.isfile(script):
        raise FileNotFoundError(script)
    py = os.path.join(root, ".venv", "bin", "python3")
    if not os.path.isfile(py):
        py = os.path.join(root, ".venv", "bin", "python")
    if not os.path.isfile(py):
        py = shutil.which("python3") or sys.executable
    cmd = [py, script, "--incremental-merge"] + list(extra)
    print("Command:", " ".join(cmd))
    subprocess.call(cmd, cwd=root, env=os.environ)


try:
    _ = STATES_MODE, STATES_CSV, LOAD_TYPES, COUNTY_GAP_DISCOVERY, IGNORE_CHECKPOINT
except NameError as e:
    raise RuntimeError("Run step **3 · Configure** before this cell.") from e

_ex = build_wikidata_extra_args(
    STATES_MODE,
    STATES_CSV,
    tuple(LOAD_TYPES),
    COUNTY_GAP_DISCOVERY,
    IGNORE_CHECKPOINT,
)
launch_loader(_ex)

Command: /home/developer/projects/open-navigator/.venv/bin/python3 /home/developer/projects/open-navigator/scripts/datasources/wikidata/load_jurisdictions_wikidata.py --types city --no-retry-county-gap-states --no-priority-states --no-all-us-states --states AL --no-force


2026-05-09 07:28:50.821 | INFO     | __main__:_load:347 - Resuming from checkpoint: 1 tasks already done
2026-05-09 07:28:50.828 | INFO     | scripts.datasources.wikidata.wikidata_integration:__init__:168 - WIKIDATA_USER_AGENT_POOL enabled: rotating 3 User-Agent value(s)
2026-05-09 07:28:50.829 | INFO     | scripts.datasources.wikidata.wikidata_integration:__init__:172 - UA net stats logging: every 10 WDQS completions (WIKIDATA_LOG_UA_STATS / WIKIDATA_UA_STATS_LOG_EVERY_NET)
2026-05-09 07:28:50.829 | INFO     | __main__:load_state:1846 - 
2026-05-09 07:28:50.829 | INFO     | __main__:load_state:1847 - ================================================================================
2026-05-09 07:28:50.829 | INFO     | __main__:load_state:1848 - LOADING WIKIDATA FOR Alabama
2026-05-09 07:28:50.829 | INFO     | __main__:load_state:1849 - ================================================================================
2026-05-09 07:28:50.850 | INFO     | __main__:_query_cities_in_state:874

## 7 · CLI help

In [ ]:
import os, shutil, subprocess, sys
py = shutil.which("python3") or sys.executable
subprocess.run([py, "scripts/datasources/wikidata/load_jurisdictions_wikidata.py", "--help"], cwd=os.getcwd())